# 06 — Fusion Variants: M3, M4, M5, M6

Trains four architecture variants sequentially, each building on the previous:

| Model | Change from previous | Config |
|---|---|---|
| M3 | +FEM+POPAM (no depth) | m3_fem_popam.yaml |
| M4 | +EarlyDepth (depth as 4th channel) | m4_early_depth.yaml |
| M5 | +LateDepth (WBF post-processing) | m5_late_depth.yaml |
| M6 | +CrossAttnDepth (P4 only) | m6_xattn_depth.yaml |

**Documented negative result for M6:**
Test cross-attention at P3 (80×80=6400 tokens) ONCE — record the OOM memory usage, then document it in `results/narratives/M6.md`.

**Prerequisites:**
- Notebook `05_augmentation` gate must have passed (M2 > M1 + 1.0 AP_hard)
- Notebook `04_depth_precomputation` must have completed
- `aug_probability` updated to optimal p in M3–M6 configs

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.logger import ExperimentLogger
from src.metrics import compute_kitti_ap, sample_to_annotation
from src.plotting import create_narrative_template

ROOT = Path('..').resolve()
TABLES_DIR = ROOT / 'results' / 'tables'
NARRATIVES_DIR = ROOT / 'results' / 'narratives'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
NARRATIVES_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = 'local'   # change to 'kaggle' for full training
SMOKE_TEST = (RUN_MODE == 'local')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}  |  Mode: {RUN_MODE}')

In [ ]:
def train_and_evaluate(config_path, model_id, extra_overrides=None):
    """Train a model variant and return (ap_easy, ap_mod, ap_hard, cfg)."""
    from ultralytics import YOLO

    overrides = {
        'run_mode': RUN_MODE,
        'epochs': 3 if SMOKE_TEST else 100,
        'data_limit': 100 if SMOKE_TEST else None,
    }
    if extra_overrides:
        overrides.update(extra_overrides)

    cfg = load_config(config_path, overrides=overrides)
    cfg.ensure_dirs()
    set_all_seeds(cfg.seed)

    val_ds = KITTIDataset(cfg.kitti_root, split='val', imgsz=cfg.imgsz)
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch, shuffle=False,
        collate_fn=collate_fn, num_workers=0
    )

    model = YOLO('yolov8s.pt')
    logger = ExperimentLogger(
        run_id=f'{model_id}_seed{cfg.seed}',
        log_dir=cfg.log_dir,
        config=vars(cfg),
    )

    with logger:
        model.train(
            data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),
            epochs=cfg.epochs,
            imgsz=cfg.imgsz,
            batch=cfg.batch,
            optimizer=cfg.optimizer,
            lr0=cfg.lr0,
            lrf=cfg.lrf,
            momentum=cfg.momentum,
            weight_decay=cfg.weight_decay,
            warmup_epochs=cfg.warmup_epochs,
            amp=cfg.amp,
            seed=cfg.seed,
            project=str(cfg.checkpoint_dir),
            name=model_id,
            exist_ok=True,
            device=device,
        )

        best_ckpt = cfg.checkpoint_dir / model_id / 'weights' / 'best.pt'
        best_model = YOLO(str(best_ckpt))
        best_model.model.eval()

        predictions, annotations = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(device)
                results = best_model(imgs, verbose=False)
                for i, r in enumerate(results):
                    boxes = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    scores = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    predictions.append({'image_id': batch['image_id'][i], 'boxes': boxes, 'scores': scores})
                    annotations.append(sample_to_annotation(batch, i))

        ap_easy = compute_kitti_ap(predictions, annotations, 'easy')
        ap_mod  = compute_kitti_ap(predictions, annotations, 'moderate')
        ap_hard = compute_kitti_ap(predictions, annotations, 'hard')
        logger.log_metrics({'AP_easy': ap_easy, 'AP_mod': ap_mod, 'AP_hard': ap_hard})

    create_narrative_template(model_id, NARRATIVES_DIR)
    print(f'{model_id}  AP_easy={ap_easy:.2f}  AP_mod={ap_mod:.2f}  AP_hard={ap_hard:.2f}')
    return ap_easy, ap_mod, ap_hard, cfg

In [ ]:
# ── M3: +FEM+POPAM ────────────────────────────────────────────────────────────
print('='*60 + '\nM3: +FEM+POPAM (no depth fusion)\n' + '='*60)
m3_easy, m3_mod, m3_hard, m3_cfg = train_and_evaluate(
    ROOT / 'configs' / 'm3_fem_popam.yaml', 'M3'
)

In [ ]:
# ── M4: +EarlyDepth ───────────────────────────────────────────────────────────
print('='*60 + '\nM4: +EarlyDepth (depth as 4th input channel)\n' + '='*60)
m4_easy, m4_mod, m4_hard, m4_cfg = train_and_evaluate(
    ROOT / 'configs' / 'm4_early_depth.yaml', 'M4'
)

In [ ]:
# ── M5: +LateDepth ────────────────────────────────────────────────────────────
print('='*60 + '\nM5: +LateDepth (WBF post-processing, RGB=0.7, depth=0.3)\n' + '='*60)
m5_easy, m5_mod, m5_hard, m5_cfg = train_and_evaluate(
    ROOT / 'configs' / 'm5_late_depth.yaml', 'M5'
)

In [ ]:
# ── M6: +CrossAttnDepth ───────────────────────────────────────────────────────
print('='*60 + '\nM6: +CrossAttnDepth (P4 only, confidence-map-gated)\n' + '='*60)
m6_easy, m6_mod, m6_hard, m6_cfg = train_and_evaluate(
    ROOT / 'configs' / 'm6_xattn_depth.yaml', 'M6'
)

In [ ]:
# ── Documented negative result: P3 cross-attention OOM ───────────────────────
# Test P3 attention ONCE, record peak GPU memory, then document.
# This is intentionally expected to OOM at batch=16 with 8 heads.

print('\nDocumented negative result: testing P3 cross-attention (expected OOM)...')

try:
    import gc
    from src.fusion import CrossAttentionFusion

    torch.cuda.empty_cache()
    gc.collect()

    # Simulate P3 attention: 80x80 = 6400 tokens, batch=16, channels=128
    B, C, H, W = 16, 128, 80, 80
    if torch.cuda.is_available():
        before_mem = torch.cuda.memory_allocated() / 1e9
        rgb_p3   = torch.randn(B, C, H, W, device='cuda')
        depth_p3 = torch.randn(B, C, H, W, device='cuda')

        from src.fusion import MultiHeadCrossAttention
        xattn_p3 = MultiHeadCrossAttention(C, num_heads=8).to('cuda')

        try:
            _ = xattn_p3(rgb_p3, depth_p3)
            peak_mem = torch.cuda.max_memory_allocated() / 1e9
            print(f'  P3 attention completed (unexpected). Peak memory: {peak_mem:.2f} GB')
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                peak_mem = torch.cuda.max_memory_allocated() / 1e9
                print(f'  [EXPECTED OOM] P3 attention failed as expected.')
                print(f'  Peak GPU memory before OOM: {peak_mem:.2f} GB')
                print(f'  Attention matrix size: {H*W}x{H*W} = {H*W*H*W:,} elements per head')
                oom_note = (
                    f'P3 cross-attention OOM test result:\n'
                    f'  Sequence length: {H*W} tokens (80x80)\n'
                    f'  Attention matrix: {H*W}x{H*W} = {H*W*H*W:,} elements/head\n'
                    f'  Batch={B}, heads=8\n'
                    f'  Peak GPU memory: {peak_mem:.2f} GB → OOM\n'
                    f'  P4 (40x40=1600 tokens) used instead.'
                )
                # Append to M6 narrative
                m6_narrative = NARRATIVES_DIR / 'M6.md'
                if m6_narrative.exists():
                    with open(m6_narrative, 'a') as f:
                        f.write(f'\n## Documented Negative Result: P3 Cross-Attention OOM\n\n```\n{oom_note}\n```\n')
                    print(f'  OOM note appended to {m6_narrative}')
            else:
                raise
    else:
        print('  [SKIP] No CUDA device — P3 OOM test requires GPU.')
except Exception as e:
    print(f'  P3 OOM test error: {e}')

In [ ]:
# ── Append M3–M6 to results table ─────────────────────────────────────────────
results_csv = TABLES_DIR / 'val_results_all_models.csv'

new_rows = [
    {'model_id': 'M3', 'description': '+FEM+POPAM (no depth)',
     'AP_easy': m3_easy, 'AP_mod': m3_mod, 'AP_hard': m3_hard,
     'aug_p': m3_cfg.aug_probability, 'use_fem': True, 'use_popam': True,
     'fusion_type': 'none', 'use_vis_head': False,
     'seed': m3_cfg.seed, 'epochs': m3_cfg.epochs},
    {'model_id': 'M4', 'description': '+EarlyDepth (4th channel)',
     'AP_easy': m4_easy, 'AP_mod': m4_mod, 'AP_hard': m4_hard,
     'aug_p': m4_cfg.aug_probability, 'use_fem': True, 'use_popam': True,
     'fusion_type': 'early', 'use_vis_head': False,
     'seed': m4_cfg.seed, 'epochs': m4_cfg.epochs},
    {'model_id': 'M5', 'description': '+LateDepth (WBF)',
     'AP_easy': m5_easy, 'AP_mod': m5_mod, 'AP_hard': m5_hard,
     'aug_p': m5_cfg.aug_probability, 'use_fem': True, 'use_popam': True,
     'fusion_type': 'late', 'use_vis_head': False,
     'seed': m5_cfg.seed, 'epochs': m5_cfg.epochs},
    {'model_id': 'M6', 'description': '+CrossAttnDepth (P4, confidence-gated)',
     'AP_easy': m6_easy, 'AP_mod': m6_mod, 'AP_hard': m6_hard,
     'aug_p': m6_cfg.aug_probability, 'use_fem': True, 'use_popam': True,
     'fusion_type': 'cross_attention', 'use_vis_head': False,
     'seed': m6_cfg.seed, 'epochs': m6_cfg.epochs},
]

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df = df[~df['model_id'].isin(['M3', 'M4', 'M5', 'M6'])]
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
else:
    df = pd.DataFrame(new_rows)

df.to_csv(results_csv, index=False)
print(f'Saved to: {results_csv}')
df